In [1]:
from pathlib import Path
import numpy as np
import nibabel as nib
import pydicom
import os
import pandas as pd

from pathlib import Path
import numpy as np
import nibabel as nib
import pandas as pd
from typing import Optional
from matplotlib.colors import Normalize, LinearSegmentedColormap
import matplotlib.pyplot as plt
import tqdm


In [24]:
paths = {
    'train': '/gpfs/data/mayerhoeferlab/projects/myeloma/datafiles_for_mars/aux_labels/train_df_with_auxiliary_labels.csv',
    'val': '/gpfs/data/mayerhoeferlab/projects/myeloma/datafiles_for_mars/aux_labels/val_df_with_auxiliary_labels.csv'
}
dfs = {split: pd.read_csv(path) for split, path in paths.items()}

In [25]:
df_seg_voxel = pd.read_csv('/gpfs/data/mayerhoeferlab/projects/myeloma/process_dicoms_for_mars/autopet3_predictions/segmentation_voxel_summary_with_excel_label.csv')

In [8]:
df_seg_voxel

,case_id,seg_voxels,excel_label
0,10002388,274,0.0
1,10024518,323,0.0
2,10032528,2235,0.0
3,10050499,67,0.0
4,10057383,78,0.0
...,...,...,...
1557,9969440,0,0.0
1558,9969540,196,0.0
1559,9971489,147,0.0
1560,9988177,5581,1.0


In [26]:
# merge df[filepath] and df_seg_voxel['case_id]
for split, df in dfs.items():
    dfs[split] = df.merge(df_seg_voxel[['case_id', 'seg_voxels']], left_on='filepath', right_on='case_id', how='left')

In [ ]:
for split, df in dfs.items():
    dfs[split]['seg_voxels'] = dfs[split]['seg_voxels'].fillna(0).astype(int) 
    dfs[split]['tumor_volume'] = dfs[split]['seg_voxels'] * dfs[split]['PET_jco']

for split, df in dfs.items():
    # now create new cols by binarizing both at quantiles of 0.25, 0.5, 0.75 using the train values
    for col in ['seg_voxels', 'tumor_volume']:
        q25 = dfs['train'][col].quantile(0.25)
        q50 = dfs['train'][col].quantile(0.5)
        q75 = dfs['train'][col].quantile(0.75)

        dfs[split][f'{col}_bin_25'] = (dfs[split][col] > q25).astype(int)
        dfs[split][f'{col}_bin_50'] = (dfs[split][col] > q50).astype(int)
        dfs[split][f'{col}_bin_75'] = (dfs[split][col] > q75).astype(int)

In [29]:
# output
for split, df in dfs.items():
    output_path = f'/gpfs/data/mayerhoeferlab/projects/myeloma/datafiles_for_mars/aux_labels/{split}_df_with_auxiliary_labels_and_tumor_volume.csv'
    df.to_csv(output_path, index=False)

In [31]:
dfs['train']

,filepath,filename,filename_header,fullfile,patient_age,patient_sex,patient_weight,smoking_status,accession,study_uid,...,bowel_uptake,case_id,seg_voxels,tumor_volume,PET_jco_bin_25,PET_jco_bin_50,PET_jco_bin_75,tumor_volume_bin_25,tumor_volume_bin_50,tumor_volume_bin_75
0,24448091,24448091_pt_2.h5,24448091_pt_2.pkl,24448091/24448091_pt_2.h5,070Y,F,98.431000,UNKNOWN,24448091,1.2.840.114350.2.232.2.798268.2.574394078.1,...,1,24448091.0,0,0,0,0,0,0,0,0
1,5445069,5445069_pt_2.h5,5445069_pt_2.pkl,5445069/5445069_pt_2.h5,052Y,M,83.536000,NO,5445069,1.3.12.2.1107.5.1.4.44843.30000010100219564928...,...,0,5445069.0,29,0,0,0,0,0,0,0
2,29248900,29248900_pt_0.h5,29248900_pt_0.pkl,29248900/29248900_pt_0.h5,048Y,M,63.502940,UNKNOWN,29248900,1.2.840.114350.2.232.2.798268.2.637997691.1,...,0,29248900.0,33,0,0,0,0,0,0,0
3,10103875,10103875_pt_0.h5,10103875_pt_0.pkl,10103875/10103875_pt_0.h5,062Y,M,59.928000,NaN,10103875,1.3.12.2.1107.5.99.1.26055.4.0.2398462621016910,...,0,10103875.0,0,0,0,0,0,0,0,0
4,14686352,14686352_pt_0.h5,14686352_pt_0.pkl,14686352/14686352_pt_0.h5,096Y,F,47.000000,NaN,14686352,1.2.840.114350.2.232.2.798268.2.222556711.1,...,0,14686352.0,3616,3616,1,1,0,1,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
734,26890228,26890228_pt_0.h5,26890228_pt_0.pkl,26890228/26890228_pt_0.h5,061Y,F,56.699053,UNKNOWN,26890228,1.2.840.114350.2.232.2.798268.2.611927056.1,...,0,26890228.0,113,113,1,1,0,1,1,0
735,11317060,11317060_pt_0.h5,11317060_pt_0.pkl,11317060/11317060_pt_0.h5,059Y,M,63.502940,UNKNOWN,11317060,1.2.840.114350.2.232.2.798268.2.122758816.1,...,0,11317060.0,170,0,0,0,0,0,0,0
736,8810026,8810026_pt_0.h5,8810026_pt_0.pkl,8810026/8810026_pt_0.h5,075Y,M,72.640000,UNKNOWN,8810026,1.2.840.114350.2.232.2.798268.2.49221970.1,...,1,8810026.0,297,297,1,1,0,1,1,1
737,17900977,17900977_pt_0.h5,17900977_pt_0.pkl,17900977/17900977_pt_0.h5,068Y,F,53.070314,UNKNOWN,17900977,1.2.840.114350.2.232.2.798268.2.283628044.1,...,0,17900977.0,0,0,0,0,0,0,0,0
